# Battle 离线训练（云端自主跑完整段）

**这个 notebook 干什么**：拿一个**整段任务包**（`task-<课>.zip` = 计划 + 课程 + 起点权重 + 动量 + 同 commit 代码 + TS 运行时），
在云机上把计划里的轮次**自主跑完**，再按你选的回传方式处置产物。全程不需要本机训练循环在线。

## 两条取包路径（cell 自动选，不用你切换）

1. **先连 hub**：`CFG.hub_url`（cloudflared 公网隧道）或 `HUB_IP`（tailnet）能通，就
   `GET /offline/task-pack?course=<课>` 直接把包取下来 —— 包就是控制台「导出任务包」写的那个文件。
2. **连不上就等你上传**：轮询工作目录找 `task-*.zip`（Colab 会弹一次上传框；Kaggle 用 **Add Data** 挂数据集）。
   等满 `CFG.wait_pack_sec`（缺省 30 分钟）还没包，cell 会**直接报错收尾**并告诉你两条下一步。
   hub 取包**最多试 `CFG.hub_tries` 轮**（缺省 10）：试满仍未拿到就转入「等上传」模式、不再轮询 hub
   ——省下的时间都留给会话；想让它继续取就重跑本 cell。

> **Kaggle 上不使用 Tailscale**（平台容器起不来隧道，且 userspace 引导会改坏平台代理）：
> 那里只走 `CFG.hub_url`（公网隧道）或手动送包，`HUB_IP` / `TS_AUTHKEY` 会被忽略。

> 所以：**全离线也行** —— 不连 hub、不上传就报错；上传了包就能跑（引导代码从包的 `code.zip` 里取）。

## 凭据（Colab Secrets / Kaggle Secrets，同名即可；也可在 CFG 里手填）

| Secret | 何时需要 | 说明 |
|---|---|---|
| `HUB_TOKEN` | 要 hub 取包或实时回传 | 与 hub 一致的 Bearer token |
| `HUB_IP` | hub 只走 tailnet 时 | 本机 Tailscale IP（裸 IP 会自动补 `CFG.hub_port`） |
| `TS_AUTHKEY` | hub 只走 tailnet 时 | Tailscale 授权 key（ephemeral 会随会话摘节点） |

`CFG.hub_url` 填了公网隧道 URL 时，**不需要 Tailscale**（前两个 Secret 也不用）。

## 产物怎么回来

- `CFG.live_backfeed = True`（缺省）：每轮跑完 best-effort 把权重/opt/指标推给 hub，
  **训练不受网络影响**（推不上去就记在本地账本，后面补；永不等、永不抛）。
- `CFG.live_backfeed = False`：跑完统一打包成 `deliver-<课>.zip`（在 Kaggle Output / Colab 文件区），
  你下载后到控制台**「导入产物」**上传 —— 导入会自动起 A 层评估。
- 两种情况都会留 `deliver-<课>.zip` 作兜底：hub 掉线时它就是唯一的路。
- **跑到一半想先拿回来**：训练 cell 停下之后跑**最后一格**（「中途取回」）——它把产物目录里
  最新的包（全量包优先，其次每轮刷新的 `LATEST.zip`）复制成 `deliver-<课>.zip` 放进下载目录，
  与跑完时**同一个名字**，控制台「导入产物」同样接受（导入后自动起 A 层评估）。

## 常用旋钮

| 键 | 缺省 | 说明 |
|---|---|---|
| `course` | — | **必填**，与控制台课程名逐字相同（取包/对账都按它）；**可给列表/逗号分隔** = 多门课按顺序串行跑完 |
| `budget_sec` | 0 | 本次最多跑多少秒（Kaggle 会话到点前干净停机的把手；0=不限） |
| `max_iters` | 0 | 本次最多再跑几轮（0=到计划末尾） |
| `device` | `auto` | `auto` → CUDA（多卡按 `use_multi_gpu`）→ TPU → CPU |
| `task_zip` | — | 已经在本机的包路径（跳过取包，直接跑） |
| `wait_pack_sec` | 1800 | 等包上限（秒） |
| `hub_tries` | 10 | hub 取包最多试几轮（试满转「等上传」模式，不再轮询 hub；0=不限） |
| `eval_on_cloud` | `false` | 云机跑 A 层评估（同 in-loop 语料，每 `eval_every` 轮）：评估在后台线程跑，但 rollout 与评估**都吃 CPU** ⇒ **本轮评估收线后才开下一轮 rollout**（最多等 5 分钟，超时只记一行、训练照常），结果随产物回本机并入账本 |
| `eval_slots` | 0 | 云机评估并发局数（0 = `max(CPU−4, CPU×0.8)`；96 核云机 ⇒ 92）；与 rollout **同一口径**，但同一时刻只有一条腿在跑 |
| `rollout_workers` | 0 | rollout 并发局数（0 = 与云机评估**同一口径** `max(CPU−4, CPU×0.8)`；非 0 时**覆盖**计划里钉着的导出机规模） |
| `eval_game_timeout_sec` | 0 | 云机评估单局超时（0 = 与 rollout 同一硬顶兜底 30s） |

## 续跑（同一个会话里再领一次任务）

重领时 hub 会递回**最新「同轮齐全」的轮次**（权重 + opt + 指标都在，回传或人工导入任一路），
云机把那一轮铺进产物目录、从它之后接着跑 —— 所以「会话被回收 → 重开 → 再跑」不会从头再来。
hub 那边没有更新的完整轮次时，就从任务包自带的起点跑（`resume: null` 是正常应答）。


In [ ]:
# @title Battle 离线训练 —— 改参数后点 Run
# 本 cell 只留：CFG / 凭据 / 保活 / 拉远端引导模块 / 从任务包内引导。
# 其余在 remote/offline_boot.py（GitHub raw；拉不到就用任务包里的 code.zip 引导）。
import contextlib, hashlib, io, os, sys, threading, time, urllib.request, zipfile
from pathlib import Path

CFG = {
    # ── 课程（取包 / 交付物都按它走；与控制台课程名逐字相同）──
    # **留空 = 自动发现**（2026-09-25）：向 hub 问 /offline/tasks 清单，可领的逐个领租约跑完
    # （用户口径：「云机不应该要在 notebook 里配置离线课程名」）；老 hub 没有清单端点会自动
    # 降级回「必须填 course」。单门课给字符串；多门课给列表（按给定顺序串行：取包/跑完/交付）。
    # 例 "c5-gae" 或 ["c5-gae", "c6-gae"]（也可写 "c5-gae, c6-gae"）
    "course": [],
    # ── 凭据一律留空：环境变量 → Colab/Kaggle Secrets → 这里手填 ──
    "hub_token": "",              # HUB_TOKEN（要实时回传 hub 或从 hub 取包时必填）
    "hub_ip": "",                 # HUB_IP：本机 hub 的 tailnet 地址（裸 IP/主机名/整条 URL）
    "hub_port": 8787,             # HUB_IP 是裸 IP/主机名时用它的端口
    "hub_url": "",                # 公网隧道 URL（cloudflared 那条）——有它就不需要 Tailscale
    "ts_authkey": "",             # TS_AUTHKEY（只在 hub 走 tailnet IP 时才需要）
    "ts_ephemeral": True,         # 会话结束自动摘节点
    "ts_engine": "",              # Tailscale 引擎顺序（缺省 "kernel,userspace"）
    # ── 任务包 ──
    # 重跑本 cell 时：① 若本机已有产物（<work>/run 三件齐）⇒ **本机优先**：不从包里导入
    # plan/manifest，包只当代码/TS 的备源，且不再为取包白等（只试一次 hub）；
    # 要改用新包重跑：先清空 battle-offline/<课>/run，或把 force_pack 置 True。
    "task_zip": "",               # 已在本机的包路径（留空则自动找 task-*.zip / 从 hub 取）
    "force_pack": False,          # True = 老行为：用包覆盖本机计划/清单（包旧就会从旧起点重跑）
    "wait_pack_sec": 1800,        # 等包上限（秒）：hub 还没导出 / 你还没上传，都在这条线上等
    "prompt_upload": True,        # Colab：等不到包时弹一次上传框（Kaggle 只能 Add Data）
    "hub_tries": 10,               # hub 取包最多试几轮（试满转「等上传」模式；0=不限）
    # ── 任务清单（2026-09-25，plan/offline-task-discovery）：course 留空时按 hub 清单跑 ──
    "auto_discover": True,        # True（且 course 留空）⇒ GET /offline/tasks；老 hub 自动降级
    "queue_mode": "drain",        # "drain" = 跑完一批继续驻守轮询；"once" = 跑一批就收工
    "queue_poll_sec": 15,         # 队列空时的轮询间隔（秒）
    "idle_wait_sec": 1800,        # 队列空最多等多久（秒）⇒ 到点收工（仅 drain）
    # 会话级驻守上限（秒，0=不限）。**与上面的 budget_sec 不同**：那个是**逐段**传给 run_loop
    # 的段预算，这个管的是「本次会话最多在队列上驻守多久」（评审 G2：两个口径不能共用一把旋钮）。
    "session_budget_sec": 0,
    # ── 训练 ──
    "live_backfeed": True,        # 实时把每轮产物 best-effort 回传 hub；关掉则跑完统一打包
    "device": "auto",             # auto → CUDA（多卡按 use_multi_gpu）→ TPU → CPU
    "use_multi_gpu": True,        # auto 且多卡时跨卡 DataParallel
    "threads": 0,                 # torch intra-op 线程（0=默认）
    "max_iters": 0,               # 本次最多再跑几轮（0=到计划末尾）
    "budget_sec": 0,              # 本次最多跑多少秒（Kaggle 会话到点前干净停机的把手）
    # ── 云机评估（A 层同口径：与 in-loop 同一份语料/行 schema/同一 wver 定义）──
    # 打开后每 eval_every 轮在本机跑一遍 A 层语料，逐局行落产物里的 eval_log.jsonl，
    # 随包回到本机由「导入产物」并进 tmp/<课程>/eval_log.jsonl（板子/门判读的就是它）。
    # 语料/难度/命数/关卡全部来自课程本身（随包的 course.jsonc）——这里只给执行面旋钮。
    "eval_on_cloud": False,       # True = 云机也跑评估（整段期间就有可信读数）
    "eval_slots": 0,              # 评估并发局数（0 = max(CPU-4, CPU*0.8)，与 rollout 同口径）
    "rollout_workers": 0,         # rollout 并发局数（0 = 同上口径；与评估真交替：本轮评估先收线）
    "eval_game_timeout_sec": 0,   # 单局超时秒（0 = 30，与 rollout 同一兜底）
    # ── 目录 ──
    "work_dir": "",               # 缺省 <download_dir>/battle-offline/<课>
    "download_dir": "",           # 交付物落点（缺省 Kaggle=/kaggle/working，Colab=/content）
    # ── 引导模块来源（GitHub raw 拉不到时的兜底已内置：用任务包里的 code.zip）──
    "repo_url": "https://github.com/HuangJian/battle.git",
    "branch": "goal-nn",
}


# 日志同时落文件：Kaggle 会话被无声回收时 cell 输出会连着丢，文件是唯一幸存者
# （AGENTS §16.2 长任务日志落文件；/kaggle/working 随 notebook 版本一并保存）。
_IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"))
_IS_COLAB = bool(os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"))
_LOG_PATHS = ["/tmp/battle-offline.log"]
if _IS_COLAB:
    _LOG_PATHS.insert(0, "/content/battle-offline.log")
if _IS_KAGGLE:
    _LOG_PATHS.insert(0, "/kaggle/working/battle-offline.log")   # 随 notebook 版本一并保存
_LOG_FILE = None
for _p in _LOG_PATHS:
    try:
        Path(_p).parent.mkdir(parents=True, exist_ok=True)
        with open(_p, "a", encoding="utf-8"):
            pass
        _LOG_FILE = Path(_p)
        break
    except OSError:
        continue


def _log(msg):
    _line = f"[{time.strftime('%H:%M:%S')}] [offline] {msg}"
    print(_line, flush=True)
    if _LOG_FILE is not None:
        try:
            with open(_LOG_FILE, "a", encoding="utf-8") as _f:
                _f.write(_line + "\n")
        except OSError:
            pass


# 引导前的平台代理环境：userspace 引导会把它换成只转发 Tailscale IP 的 tailnet 代理，
# 读平台 Secrets 时用 _platform_net_env() 临时还原（否则公网 HTTPS 必失败）。
# 2026-09-17 Kaggle 事故：引导后读 HUB_TOKEN 失败被吞成空串 → /code 401 → 会话终结。
_PROXY_KEYS = ("HTTP_PROXY", "http_proxy", "HTTPS_PROXY", "https_proxy",
               "ALL_PROXY", "all_proxy", "NO_PROXY", "no_proxy")
_PLATFORM_PROXY = {_k: os.environ.get(_k) for _k in _PROXY_KEYS}


@contextlib.contextmanager
def _platform_net_env():
    """临时还原平台自己的代理环境 —— 只给仍要访问**公网**的调用用（平台 Secrets / raw）。"""
    _saved = {_k: os.environ.get(_k) for _k in _PROXY_KEYS}
    for _k in _PROXY_KEYS:
        _orig = _PLATFORM_PROXY.get(_k)
        if _orig is None:
            os.environ.pop(_k, None)
        else:
            os.environ[_k] = _orig
    try:
        yield
    finally:
        for _k, _v in _saved.items():
            if _v is None:
                os.environ.pop(_k, None)
            else:
                os.environ[_k] = _v


def _secret(key, cfg_val=""):
    """环境变量 → Colab/Kaggle Secrets → CFG 手填（值永不进日志，只记**来源**）。"""
    def _env():
        return os.environ.get(key, "")

    def _colab():
        from google.colab import userdata
        return userdata.get(key) or ""

    def _kaggle():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key) or ""

    for _name, _get in (("env", _env), ("colab", _colab), ("kaggle", _kaggle)):
        try:
            with _platform_net_env():
                _v = _get()
        except Exception as _e:
            _log(f"凭据 {key} ← {_name} 取用失败（{type(_e).__name__}）")
            _v = ""
        if _v:
            _log(f"凭据 {key} ← {_name}")
            return str(_v).strip()
    if cfg_val:
        _log(f"凭据 {key} ← CFG 手填")
    return str(cfg_val or "").strip()


_log(
    f"course={CFG['course'] or '(未填)'} 实时回传={bool(CFG['live_backfeed'])}"
    f" 云机评估={bool(CFG.get('eval_on_cloud'))}"
    f" rollout并发={CFG.get('rollout_workers') or '自动'}"
)

# ── Keepalive ──────────────────────────────────────────────
_keepalive_stop = threading.Event()


def _keepalive_loop():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
        _sentinel = "/tmp/battle-halt-request"
        while not _keepalive_stop.is_set():
            if os.path.exists(_sentinel):
                print(f"[{time.strftime('%H:%M:%S')}] [keepalive] 检测到停机哨兵 → 释放 Colab 实例", flush=True)
                try:
                    from google.colab import runtime
                    runtime.unassign()
                except Exception as _e:
                    print(f"[{time.strftime('%H:%M:%S')}] [keepalive] unassign 失败: {_e}——请手工断开", flush=True)
                break
            try:
                from IPython.display import Javascript, display
                display(Javascript(
                    "function f(){document.querySelector('colab-connect-button')?.click();}"
                    "setTimeout(f,1000);"))
            except Exception:
                pass
            _keepalive_stop.wait(60)
    else:
        n = 0
        while not _keepalive_stop.is_set():
            print(f"[{time.strftime('%H:%M:%S')}] [keepalive] alive ({n * 5} min)", flush=True)
            n += 1
            _keepalive_stop.wait(300)


threading.Thread(target=_keepalive_loop, daemon=True, name="keepalive").start()
_log("Keepalive 已启动")
_log(f"平台={'Kaggle' if _IS_KAGGLE else ('Colab' if _IS_COLAB else '?')}"
     f" python={sys.version.split()[0]} 日志文件={_LOG_FILE or '(不可写)'}")


def _load_boot(log):
    """从 GitHub raw 取 remote/{offline_boot,tailscale_boot}.py：**每次会话都刷新**。

    取不到才回落到上一次的缓存（并响亮说明用的是旧版）——见下面那段 2026-09-22 事故注释。
    """
    dst_dir = Path("/tmp/battle-boot")
    base = CFG["repo_url"].replace("github.com", "raw.githubusercontent.com").rstrip("/")
    if base.endswith(".git"):
        base = base[:-4]
    dst_dir.mkdir(parents=True, exist_ok=True)
    # 2026-09-22 事故：旧实现是「有缓存先用缓存」，同一个 kernel 里跑过一版旧代码之后，
    # 后续每次 Run 都还在跑那份旧的——而日志只打 branch，**看不出是旧版**（实测：多课程
    # CFG 列表被旧 offline_boot 当成一门课名，拼出 "['x20-demo-mix', ...]" 这种目录名）。
    # 现在：先拉最新的（原子替换），拉不到才用缓存，并把**实际加载的那份**的 sha 打进日志。
    for _name in ("offline_boot.py", "tailscale_boot.py"):
        _dst = dst_dir / _name
        _tmp = dst_dir / (_name + ".new")
        try:
            with _platform_net_env(), urllib.request.urlopen(
                f"{base}/{CFG['branch']}/nn-training/remote/{_name}", timeout=30) as _r:
                _data = _r.read()
            if b"def " not in _data:
                raise ValueError("内容不像 Python 源文件")
            _tmp.write_bytes(_data)
            _tmp.replace(_dst)          # 原子替换：半截写入不会留下坏模块
            log(f"引导模块 {_name} 已刷新（sha12={hashlib.sha256(_data).hexdigest()[:12]}）")
        except Exception as _e:
            _tmp.unlink(missing_ok=True)
            if _dst.exists():
                log(f"拉 {_name} 失败（{type(_e).__name__}: {_e}）——用上一份缓存继续")
            else:
                log(f"拉 {_name} 失败（{type(_e).__name__}: {_e}）——改从任务包里引导")
                return None
    sys.path.insert(0, str(dst_dir))
    # ★ 2026-09-25 真机事故：上面刷新的是**磁盘**上的文件，而同 kernel 里上一次 Run 加载的
    #   `offline_boot` 还留在 `sys.modules` 里 ⇒ 下面这句 `import` 直接命中那份**旧模块**：
    #   内存跑旧代码、日志 sha 读磁盘（新版）——「看着是新的，跑的是旧的」。
    #   实测指纹：报错帧 `ensure_ts_tree` 行号 870 配的源码文本是磁盘新版 870 行的
    #   `if (root / TS_TREE_NAME).is_dir():`，而实际动作是 write_bytes（旧版 870 行）——
    #   行号与文本来自两个版本。先摘掉再导入，强制从刚刷新的文件重编译。
    for _m in ("offline_boot", "tailscale_boot", "remote.offline_boot"):
        sys.modules.pop(_m, None)
    try:
        import offline_boot
    except Exception as _e:
        log(f"导入引导模块失败（{type(_e).__name__}: {_e}）——改从任务包里引导")
        return None
    _sha12 = hashlib.sha256((dst_dir / "offline_boot.py").read_bytes()).hexdigest()[:12]
    # 内存指纹：磁盘 sha 骗得过（刷新过就是新的），模块对象骗不过（旧版没有 BOOT_SELF）。
    log(
        f"引导模块已载入（{CFG['branch']} @ sha12={_sha12}"
        f"，self={getattr(offline_boot, 'BOOT_SELF', '<missing: 早于 09-25 的旧版>')}）"
    )
    return offline_boot


def _load_boot_from_pack(log):
    """兜底：全离线也能引导 —— 任务包里的 `code.zip` 就是引导代码本身（同 commit）。

    为什么这条兜底必须在 cell 里：公网拉不到 GitHub raw 时，`task-*.zip` 是用户手上
    唯一的东西，而它自带完整源码（`remote/offline_boot.py` 也在其中）。
    """
    _cands = []
    _p = str(CFG.get("task_zip") or "").strip()
    if _p and Path(_p).is_file():
        _cands.append(Path(_p))
    for _pat in (".", "/content", "/kaggle/working", "/kaggle/input/*"):
        _base = Path(_pat)
        if _base.is_dir():
            _cands += [q for q in _base.glob("*.zip") if q.is_file()]
    _cands = sorted({q.resolve() for q in _cands},
                    key=lambda q: q.stat().st_mtime, reverse=True)
    for _z in _cands:
        try:
            with zipfile.ZipFile(_z) as _zf:
                _code = _zf.read("code.zip")
        except (KeyError, OSError, zipfile.BadZipFile):
            continue    # 不是任务包（没有 code.zip）——下一个候选
        _dst = Path("/tmp/worker-code")
        _dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(_code)) as _zz:
            _zz.extractall(_dst)
        sys.path.insert(0, str(_dst))
        from remote.offline_boot import run as _run
        log(f"引导模块来自任务包: {_z}")
        return _run
    return None


_boot = _load_boot(_log)
_run = getattr(_boot, "run", None) if _boot is not None else None
if _run is None:
    _run = _load_boot_from_pack(_log)
if _run is None:
    raise SystemExit(
        "[FATAL] 引导模块不可用：GitHub raw 拉不到，本机也没有任务包（task-*.zip）。"
        "先在控制台「导出任务包」、把它上传到本 notebook，再重跑本 cell")

# ── bun（rollout 需要 TS 运行时；run_loop 用 shutil.which('bun') 探测）──
#    必须在配置 tailnet/代理**之前**装好（装 bun 走公网 curl），故放在 _run 的 try 之前。
import shutil as _shutil, subprocess as _sp
_bun_cand = os.path.expanduser("~/.bun/bin/bun")
if not _shutil.which("bun") and not os.path.exists(_bun_cand):
    _log("安装 bun（首次 ~10s）...")
    with _platform_net_env():
        _sp.run(["bash", "-lc", "curl -fsSL https://bun.sh/install | bash"],
                check=True, capture_output=True, text=True)
os.environ["PATH"] = ((os.path.dirname(_bun_cand) + os.pathsep)
                        if os.path.exists(_bun_cand) else "") + os.environ.get("PATH", "")
_bun_bin = _shutil.which("bun") or _bun_cand
_log(f"bun 就绪：{_bun_bin} v"
     + _sp.run([_bun_bin, "--version"], capture_output=True, text=True).stdout.strip())
try:
    _rc = _run(CFG, _log, _secret, _keepalive_stop)
except SystemExit as _e:
    # 收尾原因也落文件：再按原语义上抛（SystemExit 正文不是 [offline] 行，
    # 手工复制日志时最容易被漏掉——2026-09-17 事故的第一手线索就这样丢了）。
    _log(f"SystemExit: {_e.code}")
    raise
except BaseException as _e:
    _log(f"未捕获异常 {type(_e).__name__}: {_e}")
    import traceback
    traceback.print_exc()
    raise SystemExit(-1)
_log(f"引导返回 rc={_rc}")
raise SystemExit(_rc)


In [ ]:
# @title 中途取回：把「跑到一半」的课程结果打包下载（可直接导入控制台）
# 什么时候跑：训练 cell 停下来之后 —— budget_sec 到点 / 手动 Interrupt / 会话即将被回收。
# 它把这个会话**已经落盘**的产物打成 deliver-<课>.zip（与跑完时同名同形、同一个导入路径）：
#   全量包 artifacts.zip（已收尾时才有）优先，其次 LATEST.zip（每轮 checkpoint 都刷新）。
# 训练还在跑的时候跑不了这一格（单内核被训练 cell 占着）——这是预期，不是故障。
# 本 cell 不训练、不改产物；只有本机还没有引导模块（/tmp/battle-boot、/tmp/worker-code 都没有）时
# 才会去 GitHub raw 取一次 offline_boot.py。

if "CFG" not in globals() or "_log" not in globals():
    raise SystemExit(
        "[pack] 先在同一个 kernel 里跑过上面那个训练 cell —— 本格复用它的 CFG / _log / 平台标记"
    )

# 与训练 cell 同一份引导模块（它已把 /tmp/battle-boot 或 /tmp/worker-code 插进 sys.path）。
for _p in ("/tmp/battle-boot", "/tmp/worker-code"):
    if Path(_p).is_dir() and _p not in sys.path:
        sys.path.insert(0, _p)
try:
    import offline_boot as _ob
except Exception:
    _base = CFG["repo_url"].replace("github.com", "raw.githubusercontent.com").rstrip("/")
    if _base.endswith(".git"):
        _base = _base[:-4]
    _dst = Path("/tmp/battle-boot")
    _dst.mkdir(parents=True, exist_ok=True)
    with _platform_net_env(), urllib.request.urlopen(
        f"{_base}/{CFG['branch']}/nn-training/remote/offline_boot.py", timeout=30
    ) as _r:
        _data = _r.read()
    if b"def " not in _data:
        raise SystemExit("[pack] 拉到的不是 Python 源文件 —— 检查 CFG.repo_url / CFG.branch")
    (_dst / "offline_boot.py").write_bytes(_data)
    sys.path.insert(0, str(_dst))
    import offline_boot as _ob
    _log(f"引导模块来自 GitHub raw（sha12={hashlib.sha256(_data).hexdigest()[:12]}）")

# 逐门课找 <work>/run 下最能代表当前进度的包 → 复制成 deliver-<课>.zip 放进下载目录
_packs = _ob.package_partial(CFG, _log)
if not _packs:
    _log(
        "[pack] 没打出包 —— 上面那行说了是哪个目录为空。训练还没到第一轮 checkpoint 时就是这样；"
        "产物目录也可以自己看：work_dir（缺省 <download_dir>/battle-offline/<课>）下的 run/"
    )
else:
    _log("[pack] 下载：" + "；".join(str(_q) for _q in _packs))
    _log("下载后到控制台「导入产物」上传（中途包与跑完时的 deliver-<课>.zip 同一条导入路径）")
    if _IS_COLAB:  # 训练 cell 里算好的平台标记
        try:
            from google.colab import files as _files

            for _q in _packs:
                _files.download(str(_q))
        except Exception as _e:
            _log(f"[pack] 自动下载不可用（{type(_e).__name__}: {_e}）—— 从左侧文件树取上面的路径")
    else:
        _log("[pack] Kaggle 在右侧 Output 页签下载；本地/命令行运行就直接用上面的路径")
